In [ ]:
# Celda 1
# Paths and parameters:
%cd ..

IN_JSON = "data/lod3_detalles_meshes.json"           # LOD3 with the towers and the dome of 15_volumenes and the reveals of 17_detalles
TERRAIN_JSON = "data/colegio_imperial_fotos_meshes.json"   # Only its "terreno" mesh is used (13_cubiertas)
JSON_FILE = "data/colegio_imperial_meshes.json"
BLEND_FILE = "blender/colegio_imperial_lod3.blend"
GLB_FILE = "blender/colegio_imperial_lod3.glb"
BLENDER = "C:/Program Files/Blender Foundation/Blender 5.2/blender.exe"
BPY_SCRIPT = "scripts/meshes_to_blender.py"
RENDER_SCRIPT = "scripts/render_vista.py"
ORIGIN = (440000.0, 4473700.0, 640.0)

# Colors measured on open photos and on the orthophoto, and what Street View shows (read by eye, nothing taken
# from its images): see data/materiales.csv.
MATERIALS_CSV = "data/materiales.csv"
TEXTURE_DIR = "data/texturas"
# True orthophoto rendered from the 2024 mesh of the Ayuntamiento (18_malla_referencia, vistas_malla.py), same
# box and 10 cm/px as the 2023 orthophoto (sources/ortofoto_2023/manzana_10cm.jpg), which has shadows and leaning
# walls: on it the roof around the turret read grey, but it is tiled.
ORTHO_FILE = "sources/malla_madrid_2024/malla_orto_10cm.jpg"
ORTHO_BBOX = (439958.0, 4473659.0, 440184.0, 4473885.0)   # Same as 13_cubiertas

PX = 50              # Pixels per meter of the wall textures (2 cm)
PX_ROOF = 100        # Pixels per meter of the roof textures (1 cm)
SEED = 1             # Random seed of the generated textures

COURSE = 0.46        # Height of an ashlar course (autocorrelation on the rectified photo of the instituto)
BLOCK = 1.0          # Length of an ashlar block [ESTIMADO]
TILE_COLUMN = 0.25   # Distance between rows of Arab tiles [ESTIMADO]
TILE_LENGTH = 0.40   # Visible length of one tile [ESTIMADO]
SLATE_WIDTH = 0.25   # Slate pieces [ESTIMADO]
SLATE_ROW = 0.12

# Walls of the LOD2: granite ashlar below SPLIT_Z (meters above sea level) and render above.
# Instituto: top of the granite base on Toledo (644.4 + 5.3 m, measured). Colegiata: plinth [ESTIMADO].
WALL_Z = (640.0, 705.0)   # Height range of the wall textures (from the street to the lantern)
WALLS = {
    "muro_instituto": {"split_z": 649.7, "render": "revoco_instituto"},
    # Colegiata: ochre on the low bodies (Calle de la Colegiata), white above their cornices (gable, crossing,
    # upper bodies), as in Street View and in the open photo Colegiata 15. Change height [ESTIMADO]:
    "muro_colegiata": {"split_z": 646.0, "render": "revoco_colegiata", "upper_z": 662.0, "upper": "revoco_blanco"},
    "muro_blanco": {"split_z": 646.0, "render": "revoco_blanco"},   # Towers and drum of the dome
}
INSTITUTO_PARCELS = ["0139424VK4703G", "0139423VK4703G"]
COLEGIATA_PARCEL = "0139402VK4703G"

# Roofs of the ensemble: slate where the roof reaches SLATE_Z (the two towers and the dome); grey where the
# orthophoto is grey (red minus blue < GREY_RB: zinc or flat roofs); Arab tiles everywhere else.
SLATE_Z = 685.0
GREY_RB = 20
ROOF_NORMAL_Z = 0.3   # A face is roof if its normal points up at least this much (same as 13_cubiertas)
STEP_ABOVE_BASE = 0.5 # A vertical face whose lowest point is this much above the base of its part is a roof step,
STEP_MAX_HEIGHT = 3.0 # if it is at most this tall (any height under slate: dome and towers). Taller ones are walls

# Terrain without bumps: the 2 m grid of the cloud catches cars, trees and kerbs. Nodes more than TERRAIN_OUTLIER m
# above the median within TERRAIN_RADIUS m are dropped; then each node takes the Gaussian mean (sigma TERRAIN_SIGMA m)
# of the rest. The slope of the streets stays (Calle de la Colegiata rises 3.5 m). The foot of every LOD1/LOD2 wall
# goes WALL_SINK m down, so that no gap opens where the smoothed ground is lower than the base of a building.
TERRAIN_RADIUS = 8.0
TERRAIN_OUTLIER = 0.5
TERRAIN_SIGMA = 6.0
WALL_SINK = 1.0

In [ ]:
# Celda 2
import os
import csv
import json
import math
import subprocess
import numpy as np
from PIL import Image, ImageDraw
from shapely.geometry import MultiPoint

os.makedirs(TEXTURE_DIR, exist_ok=True)
with open(MATERIALS_CSV, encoding="utf-8") as f:
    COLORS = {row["material"]: np.array([float(row["r"]), float(row["g"]), float(row["b"])]) for row in csv.DictReader(f)}
print(len(COLORS), "materials:", ", ".join(COLORS))

In [ ]:
# Celda 3
# Texture generators. Every texture repeats without seams: the noise is made in frequency space (it wraps
# around the edges) and the patterns fit a whole number of times in the tile.
rng = np.random.default_rng(SEED)

def noise(h, w, size_px):
    """Smooth random field in -1..1, with blobs of about size_px pixels, that wraps around the edges."""
    white = rng.standard_normal((h, w))
    fy = np.fft.fftfreq(h)[:, None]
    fx = np.fft.fftfreq(w)[None, :]
    blur = np.exp(-2 * (math.pi * size_px) ** 2 * (fx ** 2 + fy ** 2))   # Gaussian blur of sigma = size_px
    field = np.real(np.fft.ifft2(np.fft.fft2(white) * blur))
    return field / (np.abs(field).max() + 1e-9)

def paint(color, shade):
    """Color (r, g, b) times a brightness field (h, w): an RGB image array."""
    return np.clip(shade[:, :, None] * color[None, None, :], 0, 255)

def save(name, rgb):
    path = os.path.abspath(os.path.join(TEXTURE_DIR, name + ".png"))
    Image.fromarray(rgb.astype(np.uint8)).save(path)
    return path

def granite_grain(h, w):
    return 0.10 * noise(h, w, 0.6) + 0.04 * noise(h, w, 4) + 0.03 * noise(h, w, 30)

def ashlar(width_m, rows, px):
    """Granite ashlar: rows of blocks of BLOCK x COURSE m, every other row shifted half a block, with dark joints."""
    course_px = round(COURSE * px)
    block_px = round(BLOCK * px)
    h, w = rows * course_px, round(width_m * px)
    shade = 1 + granite_grain(h, w)
    for row in range(rows):
        shift = (block_px // 2) * (row % 2)
        for start in range(0, w, block_px):
            columns = (np.arange(start, start + block_px) + shift) % w
            shade[row * course_px:(row + 1) * course_px, columns] *= 1 + rng.normal(0, 0.05)
            shade[row * course_px:(row + 1) * course_px, columns[0]] *= 0.65       # Vertical joint
        shade[row * course_px, :] *= 0.65                                          # Horizontal joint
    return paint(COLORS["granito"], shade)

def smooth(color, size_m, px):
    """Plain surface (render, smooth stone, zinc) with soft stains."""
    n = round(size_m * px)
    shade = 1 + 0.05 * noise(n, n, 0.5 * px) + 0.015 * noise(n, n, 4) + 0.01 * noise(n, n, 0.7)
    return paint(color, shade)

def arab_tiles(size_m, px):
    """Rows of convex Arab tiles running down the slope (image columns), each tile darker at its lower end."""
    n = round(size_m * px)
    column_px = round(TILE_COLUMN * px)
    length_px = round(TILE_LENGTH * px)
    across = (np.arange(n) % column_px + 0.5) / column_px            # 0..1 across one row of tiles
    shade = np.tile(0.72 + 0.38 * np.sin(math.pi * across), (n, 1))   # Rounded tile: dark edges, light crest
    image = np.zeros((n, n, 3))
    color = COLORS["teja"]
    for start in range(0, n, column_px):
        offset = rng.integers(0, length_px)
        for top in range(-length_px, n, length_px):
            rows = (np.arange(top, top + length_px) + offset) % n
            tone = color * (1 + rng.normal(0, 0.07)) + np.array([1, 0.4, 0]) * rng.normal(0, 8)   # Some tiles more orange
            along = np.linspace(1.0, 0.8, length_px)[:, None]                                    # Shadow of the tile above
            image[np.ix_(rows, np.arange(start, start + column_px))] = (
                tone[None, None, :] * (shade[np.ix_(rows, np.arange(start, start + column_px))] * along)[:, :, None])
    image *= (1 + 0.04 * noise(n, n, 0.8))[:, :, None]
    return np.clip(image, 0, 255)

def slates(size_m, px):
    """Rectangular slates in rows, every other row shifted half a piece, with dark joints."""
    width_px = round(SLATE_WIDTH * px)
    row_px = round(SLATE_ROW * px)
    w = round(size_m * px) // width_px * width_px
    h = round(size_m * px) // row_px * row_px
    shade = 1 + 0.03 * noise(h, w, 0.8)
    for row in range(h // row_px):
        shift = (width_px // 2) * (row % 2)
        for start in range(0, w, width_px):
            columns = (np.arange(start, start + width_px) + shift) % w
            shade[row * row_px:(row + 1) * row_px, columns] *= 1 + rng.normal(0, 0.06)
            shade[row * row_px:(row + 1) * row_px, columns[0]] *= 0.6
        shade[row * row_px:row * row_px + 2, :] *= 0.6
    return paint(COLORS["pizarra"], shade), w / px, h / px

def window(w=128, h=192):
    """One window, stretched over each opening: wooden frame, two leaves with three panes, glass lighter at the top."""
    glass = COLORS["vidrio"] * np.linspace(1.5, 0.9, h)[:, None, None] * np.ones((1, w, 1))
    image = np.array(glass)
    wood = COLORS["madera"]
    frame = round(0.09 * w)
    image[:frame, :] = wood
    image[-frame:, :] = wood
    image[:, :frame] = wood
    image[:, -frame:] = wood
    image[:, w // 2 - frame // 2:w // 2 + frame // 2] = wood            # Middle of the two leaves
    for k in (1, 2):
        y = round(k * h / 3)
        image[y - 2:y + 2, :] = wood                                    # Bars between panes
    return np.clip(image, 0, 255)

def wall_atlas(split_z, render_name, upper_z=None, upper=None):
    """Wall texture as tall as WALL_Z: ashlar below split_z, render above (and another render above upper_z).
    It repeats only along the wall."""
    width_m = 4.0
    h = round((WALL_Z[1] - WALL_Z[0]) * PX)
    rows_below = math.ceil((split_z - WALL_Z[0]) / COURSE) + 1
    stone = ashlar(width_m, rows_below, PX)
    render = smooth(COLORS[render_name], width_m, PX)
    upper_render = smooth(COLORS[upper], width_m, PX) if upper else None
    image = np.zeros((h, round(width_m * PX), 3))
    for i in range(h):
        z = WALL_Z[1] - (i + 0.5) / PX                                   # Image row 0 is the top of the wall
        if z < split_z:
            image[i] = stone[int(round((split_z - z) * PX)) % stone.shape[0]]
        elif upper and z >= upper_z:
            image[i] = upper_render[i % upper_render.shape[0]]
        else:
            image[i] = render[i % render.shape[0]]
    # Shadow line under the top of the granite base:
    i_split = round((WALL_Z[1] - split_z) * PX)
    image[i_split:i_split + 2] *= 0.7
    return image, width_m

# Make and save the textures. TEXTURES[name] = (file, tile width m, tile height m); tile height None means the
# texture covers WALL_Z from bottom to top (it does not repeat upward).
TEXTURES = {}
TEXTURES["granito_sillar"] = (save("granito_sillar", ashlar(4.0, 4, PX)), 4.0, 4 * round(COURSE * PX) / PX)
TEXTURES["granito_liso"] = (save("granito_liso", smooth(COLORS["granito"], 2.0, PX) * (1 + granite_grain(100, 100))[:, :, None]), 2.0, 2.0)
for name in ["revoco_instituto", "revoco_colegiata", "revoco_blanco", "cubierta_gris"]:
    TEXTURES[name] = (save(name, smooth(COLORS[name], 4.0, PX)), 4.0, 4.0)
TEXTURES["teja"] = (save("teja", arab_tiles(2.0, PX_ROOF)), 2.0, 2.0)
slate_image, slate_w, slate_h = slates(1.0, PX_ROOF)
TEXTURES["pizarra"] = (save("pizarra", slate_image), slate_w, slate_h)
TEXTURES["ventana"] = (save("ventana", window()), None, None)          # Stretched over each face
for name, wall in WALLS.items():
    image, width_m = wall_atlas(wall["split_z"], wall["render"], wall.get("upper_z"), wall.get("upper"))
    TEXTURES[name] = (save(name, image), width_m, None)
for name, (path, w, h) in TEXTURES.items():
    print(name, Image.open(path).size, "px | tile", w, "x", h, "m")

In [ ]:
# Celda 4
# Texture coordinates, one (u, v) per face corner, in tiles of the texture (so the image repeats at its real size).
def newell_normal(points):
    nx, ny, nz = 0.0, 0.0, 0.0
    for k in range(len(points)):
        x1, y1, z1 = points[k]
        x2, y2, z2 = points[(k + 1) % len(points)]
        nx += (y1 - y2) * (z1 + z2)
        ny += (z1 - z2) * (x1 + x2)
        nz += (x1 - x2) * (y1 + y2)
    length = math.sqrt(nx * nx + ny * ny + nz * nz) or 1.0
    return nx / length, ny / length, nz / length

def wall_uv(points, texture):
    """Walls: u along the wall (horizontal), v up. Faces that look up or down: u = x, v = y."""
    _, tile_w, tile_h = TEXTURES[texture]
    nx, ny, nz = newell_normal(points)
    horizontal = math.hypot(nx, ny)
    result = []
    for x, y, z in points:
        if abs(nz) < 0.7 and horizontal > 1e-6:
            u = (-ny * x + nx * y) / horizontal
            v_m = z
        else:
            u, v_m = x, y
        if tile_h is None:   # Tall wall texture: v = 0 at WALL_Z[0], 1 at WALL_Z[1]
            v = (z + ORIGIN[2] - WALL_Z[0]) / (WALL_Z[1] - WALL_Z[0])
        else:
            v = v_m / tile_h
        result.append((u / tile_w, v))
    return result

def stretch_uv(points):
    """Window: the image covers the face once (0..1 along the wall and up)."""
    nx, ny, nz = newell_normal(points)
    horizontal = math.hypot(nx, ny)
    if abs(nz) < 0.7 and horizontal > 1e-6:
        coords = [((-ny * x + nx * y) / horizontal, z) for x, y, z in points]
    else:
        coords = [(x, y) for x, y, z in points]
    us, vs = [c[0] for c in coords], [c[1] for c in coords]
    du, dv = (max(us) - min(us)) or 1.0, (max(vs) - min(vs)) or 1.0
    return [((u - min(us)) / du, (v - min(vs)) / dv) for u, v in coords]

def main_directions(vertices):
    """The four horizontal directions of the sides of the smallest rectangle around a roof."""
    rectangle = MultiPoint([(x, y) for x, y, z in vertices]).minimum_rotated_rectangle
    (x1, y1), (x2, y2) = list(rectangle.exterior.coords)[:2]
    angle = math.atan2(y2 - y1, x2 - x1)
    return [(math.cos(angle + k * math.pi / 2), math.sin(angle + k * math.pi / 2)) for k in range(4)]

def roof_uv(points, texture, directions):
    """Roofs: v runs down the slope, u along the eaves. The slope direction is rounded to the nearest side of the
    roof, so that all the faces of one pitch share it and the tiles run on without breaks (the 0.5 m grid is noisy).
    v is measured on the plan, not on the slope: the tiles look up to 1/cos(slope) longer [simplification]."""
    _, tile_w, tile_h = TEXTURES[texture]
    nx, ny, nz = newell_normal(points)
    if math.hypot(nx, ny) < 0.1:
        dx, dy = directions[0]
    else:
        dx, dy = max(directions, key=lambda d: d[0] * nx + d[1] * ny)
    return [((-dy * x + dx * y) / tile_w, (dx * x + dy * y) / tile_h) for x, y, z in points]

In [ ]:
# Celda 5
# Roof material of each LOD2 part of the ensemble, from its height and the color of the orthophoto under it:
ortho = Image.open(ORTHO_FILE)
ortho_pixels = np.asarray(ortho).astype(float)
W, H = ortho.size
xmin, ymin, xmax, ymax = ORTHO_BBOX

def ortho_median(vertices, faces):
    mask = Image.new("L", (W, H), 0)
    draw = ImageDraw.Draw(mask)
    for face in faces:
        corners = []
        for i in face:
            x, y, z = vertices[i]
            corners.append(((x + ORIGIN[0] - xmin) / (xmax - xmin) * W, (1 - (y + ORIGIN[1] - ymin) / (ymax - ymin)) * H))
        draw.polygon(corners, fill=1)
    inside = np.asarray(mask).astype(bool)
    return np.median(ortho_pixels[inside], axis=0) if inside.sum() > 0 else None

def roof_material(part, roof_faces):
    top = max(z for x, y, z in part["vertices"]) + ORIGIN[2]
    if top >= SLATE_Z:
        return "pizarra"
    color = ortho_median(part["vertices"], roof_faces)
    if color is not None and color[0] - color[2] < GREY_RB:
        return "cubierta_gris"
    return "teja"

def wall_material(part, roof):
    if part["parcel"] in INSTITUTO_PARCELS:
        return "muro_instituto"
    if roof == "pizarra":
        return "muro_blanco"
    return "muro_colegiata"

# Material of each LOD3 element, from its name and the material column of its table (data/fachadas/*.csv):
def element_material(name, group):
    if group == "piedra":
        if "sillares" in name or "zocalo_granito" in name or name.startswith("zocalo_norte"):
            return "granito_sillar"
        return "granito_liso"
    if group == "vidrio":
        if (("hueco" in name or "ventana" in name) and "hornacina" not in name) or name.startswith("balc"):
            return "ventana"
        return "sombra"   # Arches of the cloister and the niche: dark
    if group == "metal":
        return "hierro"
    if group == "pizarra":
        return "pizarra"
    return "revoco_colegiata"   # Body of the mirador on Calle de la Colegiata

In [ ]:
# Celda 6
# Split every LOD1 / LOD2 building into roof and walls (as in 13_cubiertas) and give each piece its material:
def sub_mesh(mesh, faces):
    new_index = {}
    vertices = []
    new_faces = []
    for face in faces:
        new_face = []
        for i in face:
            if i not in new_index:
                new_index[i] = len(vertices)
                vertices.append(mesh["vertices"][i])
            new_face.append(new_index[i])
        new_faces.append(tuple(new_face))
    part = {k: v for k, v in mesh.items() if k not in ("uv", "image")}
    part["vertices"] = vertices
    part["faces"] = new_faces
    return part

def with_texture(part, texture, uv_function, *extra):
    part["texture"] = TEXTURES[texture][0]
    part["material"] = texture
    part["loop_uv"] = [uv_function([part["vertices"][i] for i in face], *extra) for face in part["faces"]]
    return part

def with_color(part, name):
    part["color"] = (name, *[int(c) for c in COLORS[name]])
    part["material"] = name
    return part

with open(IN_JSON, encoding="utf-8") as f:
    meshes = json.load(f)["parts"]

# Sink the foot of the walls:
for mesh in meshes:
    if mesh.get("lod") in (1, 2):
        base = min(z for x, y, z in mesh["vertices"])
        mesh["vertices"] = [(x, y, z - WALL_SINK if z < base + 0.01 else z) for x, y, z in mesh["vertices"]]

result = []
count = {}
for mesh in meshes:
    if mesh.get("lod") in (1, 2):
        roof_faces, other_faces = [], []
        for face in mesh["faces"]:
            nz = newell_normal([mesh["vertices"][i] for i in face])[2]
            (roof_faces if nz >= ROOF_NORMAL_Z else other_faces).append(face)
        ensemble = mesh["group"] == "conjunto"
        roof = roof_material(mesh, roof_faces) if ensemble and roof_faces else None
        # In the ensemble, a low vertical face that does not reach down to the base of the part is a step inside the
        # roof (the "curtains" of the 2.5D grid: dome, towers, nave): it takes the roof material, not the wall's.
        base = min(z for x, y, z in mesh["vertices"])
        steps = []
        if ensemble and roof:
            def is_roof_step(face):
                heights = [mesh["vertices"][i][2] for i in face]
                return min(heights) > base + STEP_ABOVE_BASE and (roof == "pizarra" or max(heights) - min(heights) <= STEP_MAX_HEIGHT)
            is_step = [is_roof_step(face) for face in other_faces]
            steps = [face for face, step in zip(other_faces, is_step) if step]
            other_faces = [face for face, step in zip(other_faces, is_step) if not step]
        if other_faces:
            walls = sub_mesh(mesh, other_faces)
            if ensemble:
                result.append(with_texture(walls, wall_material(mesh, roof), wall_uv, wall_material(mesh, roof)))
            else:
                result.append(with_color(walls, "contexto_muro"))
        if steps:
            step_part = sub_mesh(mesh, steps)
            step_part["name"] = mesh["name"] + "_escalones"
            result.append(with_texture(step_part, roof, wall_uv, roof))
        if roof_faces:
            top = sub_mesh(mesh, roof_faces)
            top["name"] = mesh["name"] + "_cubierta"
            if ensemble:
                result.append(with_texture(top, roof, roof_uv, roof, main_directions(top["vertices"])))
            else:
                result.append(with_color(top, "contexto_cubierta"))
    else:
        # Pieces of 15_volumenes carry their material; the rest is read from the name and group:
        material = mesh.get("material_hint") or element_material(mesh["name"], mesh["group"])
        part = sub_mesh(mesh, mesh["faces"])
        if material == "ventana":
            result.append(with_texture(part, material, stretch_uv))
        elif material in TEXTURES:
            result.append(with_texture(part, material, wall_uv, material))
        else:
            result.append(with_color(part, material))

# Terrain of 13_cubiertas, smoothed (see TERRAIN_* in Celda 1), in plain grey:
with open(TERRAIN_JSON, encoding="utf-8") as f:
    terrain = [m for m in json.load(f)["parts"] if m["name"] == "terreno"][0]
points = np.array(terrain["vertices"])
xy, z = points[:, :2], points[:, 2]
CHUNK = 500
local_median = np.empty(len(points))
for start in range(0, len(points), CHUNK):
    near = ((xy[start:start + CHUNK, None, :] - xy[None, :, :]) ** 2).sum(-1) <= TERRAIN_RADIUS ** 2
    local_median[start:start + CHUNK] = [np.median(z[row]) for row in near]
keep = z <= local_median + TERRAIN_OUTLIER
smoothed = np.empty(len(points))
for start in range(0, len(points), CHUNK):
    weight = np.exp(-((xy[start:start + CHUNK, None, :] - xy[None, keep, :]) ** 2).sum(-1) / (2 * TERRAIN_SIGMA ** 2))
    smoothed[start:start + CHUNK] = weight @ z[keep] / weight.sum(1)
print(int((~keep).sum()), "of", len(points), "terrain nodes dropped as bumps | change: median",
      round(float(np.median(smoothed - z)), 2), "m, 5-95 %", np.percentile(smoothed - z, [5, 95]).round(2), "m")
terrain["vertices"] = [(x, y, float(h)) for (x, y, _), h in zip(terrain["vertices"], smoothed)]
result.append(with_color(sub_mesh(terrain, terrain["faces"]), "terreno"))

for part in result:
    count[part["material"]] = count.get(part["material"], 0) + 1
print(len(result), "meshes")
for name in sorted(count):
    print(" ", name, count[name])

In [ ]:
# Celda 7
# Save and run Blender:
with open(JSON_FILE, "w", encoding="utf-8") as f:
    json.dump({"origin": ORIGIN, "parts": result}, f)

command = [BLENDER, "-b", "--factory-startup", "--python", BPY_SCRIPT, "--", JSON_FILE, BLEND_FILE, GLB_FILE]
run = subprocess.run(command, capture_output=True, text=True)
for line_text in run.stdout.splitlines():
    if "Objects" in line_text or "Error" in line_text or "Saved" in line_text:
        print(line_text)
print("Return code:", run.returncode)
# Blender exits with 0 even when the script fails: check that it saved
if run.returncode != 0 or "Saved" not in run.stdout:
    print(run.stdout[-2000:], run.stderr[-2000:])
    raise RuntimeError("Blender did not save the model")
for path in [BLEND_FILE, GLB_FILE]:
    print(path, round(os.path.getsize(path) / 1e6, 2), "MB")

In [ ]:
# Celda 8
# Views: from Calle de Toledo (south-west, above the street) and from the air (south-west).
PORTAL_VIEW = ((-31.9, 105.8, 6.1), (-19.6, 111.1, 9.0), 30)       # From the sidewalk across Toledo, a little to the south
COLEGIATA_VIEW = ((-40.6, 138.4, 6.1), (-16.9, 134.9, 15.0), 28)   # Across Toledo, in front of the portal
CLOISTER_VIEW = ((9.5, 92.0, 5.9), (-4.4, 107.5, 10.0), 18)        # From the SE corner of the courtyard
VIEWS = {
    "renders/vista_general_materiales.jpg": ((-95, 25, 75), (25, 105, 12), 35),
    "renders/vista_aerea_materiales.jpg": ((-60, -40, 170), (30, 105, 10), 35),
    "renders/vista_toledo_materiales.jpg": ((-40, 70, 6.5), (0, 112, 20), 24),
    "renders/vista_cupula_torres.jpg": ((-62, 178, 52), (10, 132, 40), 38),
    # Close views to check the details against the photos (17_detalles): portal 1 of the instituto, the Colegiata
    # front, and the cloister.
    "renders/detalle_portada_instituto.jpg": PORTAL_VIEW,
    "renders/detalle_colegiata.jpg": COLEGIATA_VIEW,
    "renders/detalle_claustro.jpg": CLOISTER_VIEW,
}
for out, (camera, target, lens) in VIEWS.items():
    command = [BLENDER, "-b", BLEND_FILE, "--python", RENDER_SCRIPT, "--", os.path.abspath(out),
               *[str(v) for v in camera], *[str(v) for v in target], str(lens), "2000", "1400"]
    run = subprocess.run(command, capture_output=True, text=True)
    print(out, "| return code", run.returncode)
    if run.returncode != 0:
        print(run.stderr[-1500:])